# 07 – SQL cross-check of the key results

The main results of questions 2, 3 and 4 were calculated in pandas. Here they are **calculated again, independently, in SQL**, and the two answers are compared. If they match, it is unlikely that a mistake in the Python code changed the conclusions.

The queries are in the [`sql/`](../sql) folder and run with **DuckDB**, which can query CSV files directly without setting up a database server. The SQL starts from the clean tables of notebook 02 (`province_analysis_table.csv` and `national_monthly.csv`), not from the outputs of notebooks 04–06, so each result is rebuilt from the same starting point.

| File | Question | SQL used |
| --- | --- | --- |
| `01_national_sockets.sql` | 2 – national trend | simple ratio per month |
| `02_region_summary.sql` | 2 – regions | `GROUP BY` with sums |
| `03_hdi_groups.sql` | 3 – development groups | `NTILE(5)` window function |
| `04_priority_score.sql` | 4 – priority ranking | `PERCENT_RANK` window function, CTEs |

In [1]:
import duckdb
import pandas as pd

PROCESSED = "../data/processed/"
SQL = "../sql/"

con = duckdb.connect()
# Register each clean CSV as a table name that the SQL files can use
tables = {
    "provinces": "province_analysis_table.csv",
    "national": "national_monthly.csv",
}
for name, file in tables.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('{PROCESSED}{file}')")


def run_sql(file):
    query = open(SQL + file).read()
    print(query)
    return con.sql(query).df()

## 1. National trend (question 2)

In [2]:
national_sql = run_sql("01_national_sockets.sql")
national_sql

-- Question 2: electric cars per public charging socket, Türkiye, every month with socket data
SELECT
    month,
    cars_electric,
    sockets_total,
    sockets_dc,
    ROUND(cars_electric / sockets_total, 1) AS ev_per_socket,
    ROUND(cars_electric / sockets_dc, 1) AS ev_per_dc_socket
FROM national
WHERE sockets_total IS NOT NULL
ORDER BY month;



,month,cars_electric,sockets_total,sockets_dc,ev_per_socket,ev_per_dc_socket
0,2025-09,319155.0,35002.0,15010.0,9.1,21.3
1,2025-10,332010.0,36280.0,15603.0,9.2,21.3
2,2025-11,348908.0,37473.0,16051.0,9.3,21.7
3,2025-12,370591.0,38808.0,16713.0,9.5,22.2
4,2026-01,385876.0,39694.0,17059.0,9.7,22.6
5,2026-02,395697.0,40575.0,17433.0,9.8,22.7
6,2026-03,408314.0,41938.0,18038.0,9.7,22.6
7,2026-04,423793.0,43009.0,18541.0,9.9,22.9
8,2026-05,436474.0,44175.0,19050.0,9.9,22.9
9,2026-06,445939.0,45097.0,19663.0,9.9,22.7


In [3]:
# Same calculation in pandas (as in notebook 04)
national = pd.read_csv(PROCESSED + "national_monthly.csv").dropna(subset=["sockets_total"]).set_index("month")
national_pandas = pd.DataFrame({
    "ev_per_socket": (national["cars_electric"] / national["sockets_total"]).round(1),
    "ev_per_dc_socket": (national["cars_electric"] / national["sockets_dc"]).round(1),
})
columns = ["ev_per_socket", "ev_per_dc_socket"]
print("All months match:", (national_sql.set_index("month")[columns].reindex(national_pandas.index) == national_pandas[columns]).all().all())

All months match: True


## 2. Region summary (question 2)

In [4]:
region_sql = run_sql("02_region_summary.sql")
region_sql

-- Question 2: public charging stations per 10,000 cars and per 100,000 people, by İBBS-1 region
SELECT
    region,
    SUM(public_stations) AS public_stations,
    SUM(cars) AS cars,
    ROUND(SUM(public_stations) / SUM(cars) * 10000, 1) AS stations_per_10k_cars,
    ROUND(SUM(public_stations) / SUM(population_2025) * 100000, 1) AS stations_per_100k_people
FROM provinces
GROUP BY region
ORDER BY stations_per_10k_cars;



,region,public_stations,cars,stations_per_10k_cars,stations_per_100k_people
0,Batı Anadolu,1608.0,2809480.0,5.7,18.9
1,Akdeniz,1511.0,2455177.0,6.2,13.7
2,Batı Karadeniz,589.0,930636.0,6.3,12.5
3,İstanbul,2742.0,4210192.0,6.5,17.4
4,Ege,1832.0,2581295.0,7.1,16.6
5,Güneydoğu Anadolu,553.0,725668.0,7.6,5.8
6,Doğu Marmara,1513.0,1822019.0,8.3,17.3
7,Kuzeydoğu Anadolu,150.0,173288.0,8.7,7.1
8,Doğu Karadeniz,382.0,436094.0,8.8,14.1
9,Orta Anadolu,748.0,844201.0,8.9,18.0


In [5]:
# Same calculation in pandas (as in notebook 04)
provinces = pd.read_csv(PROCESSED + "province_analysis_table.csv")
totals = provinces.groupby("region")[["public_stations", "cars", "population_2025"]].sum()
region_pandas = pd.DataFrame({
    "stations_per_10k_cars": (totals["public_stations"] / totals["cars"] * 10_000).round(1),
    "stations_per_100k_people": (totals["public_stations"] / totals["population_2025"] * 100_000).round(1),
})
columns = ["stations_per_10k_cars", "stations_per_100k_people"]
print("All regions match:", (region_sql.set_index("region")[columns].reindex(region_pandas.index) == region_pandas[columns]).all().all())

# Türkiye totals used in notebooks 04-06
print("Public stations in the 81 provinces:", con.sql("SELECT SUM(public_stations) FROM provinces").fetchone()[0])
print("Cars in the 81 provinces:", con.sql("SELECT SUM(cars) FROM provinces").fetchone()[0])

All regions match: True
Public stations in the 81 provinces: 12736
Cars in the 81 provinces: 18115875


## 3. Development groups (question 3)

`NTILE(5)` sorts the provinces by HDI and splits them into five groups of (almost) equal size, the same as `pd.qcut` in notebook 05. With 81 provinces, both put 17 provinces in the first group and 16 in each of the others.

In [6]:
groups_sql = run_sql("03_hdi_groups.sql")
groups_sql

-- Question 3: five development groups by Province Human Development Index (2022)

-- NTILE(5) sorts the provinces by HDI and splits them into five groups of (almost) equal size
WITH grouped AS (
    SELECT
        *,
        NTILE(5) OVER (ORDER BY hdi_2022) AS hdi_group
    FROM provinces
)

SELECT
    hdi_group,
    COUNT(*) AS provinces,
    ROUND(SUM(cars) / SUM(population_2025) * 1000, 1) AS cars_per_1000_people,
    ROUND(SUM(public_stations) / SUM(population_2025) * 100000, 1) AS stations_per_100k_people,
    ROUND(SUM(public_stations) / SUM(cars) * 10000, 1) AS stations_per_10k_cars
FROM grouped
GROUP BY hdi_group
ORDER BY hdi_group;



,hdi_group,provinces,cars_per_1000_people,stations_per_100k_people,stations_per_10k_cars
0,1,17,65.9,6.3,9.6
1,2,16,161.6,8.6,5.3
2,3,16,200.8,13.9,6.9
3,4,16,214.9,17.2,8.0
4,5,16,268.8,18.6,6.9


In [7]:
# Same calculation in pandas (as in notebook 05)
provinces["hdi_group"] = pd.qcut(provinces["hdi_2022"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
totals = provinces.groupby("hdi_group")[["public_stations", "cars", "population_2025"]].sum()
groups_pandas = pd.DataFrame({
    "cars_per_1000_people": (totals["cars"] / totals["population_2025"] * 1000).round(1),
    "stations_per_100k_people": (totals["public_stations"] / totals["population_2025"] * 100_000).round(1),
    "stations_per_10k_cars": (totals["public_stations"] / totals["cars"] * 10_000).round(1),
})
columns = list(groups_pandas.columns)
print("Same group sizes:", groups_sql.set_index("hdi_group")["provinces"].tolist(),
      provinces["hdi_group"].value_counts().sort_index().tolist())
print("All groups match:", (groups_sql.set_index("hdi_group")[columns].reindex(groups_pandas.index) == groups_pandas[columns]).all().all())

Same group sizes: [17, 16, 16, 16, 16] [17, 16, 16, 16, 16]
All groups match: True


## 4. Priority score (question 4)

The SQL version uses `PERCENT_RANK`, the standard SQL percentile rank. It is calculated as (rank − 1) / (number of rows − 1), while pandas uses rank / number of rows. Both measure the same position on a slightly different scale, so a few scores can differ in the second decimal place, but **the order of provinces should be the same**.

Some provinces have exactly the same score (e.g. the three ranks of Sinop, Ordu and Zonguldak add up to the same total). Both sides treat these the same way: `RANK()` in SQL and `rank(method="min")` in pandas give tied provinces the same place.

In [8]:
top20_sql = run_sql("04_priority_score.sql")
top20_sql

-- Question 4: priority score for public charging investment, top 20 provinces

-- The three indicators
WITH indicators AS (
    SELECT
        province,
        region,
        public_stations / cars * 10000 AS stations_per_10k_cars,
        no_parking_share_2021,
        new_cars_2026_jan_sep / population_2025 * 1000 AS new_cars_per_1000_people
    FROM provinces
),

-- Percentile rank of each indicator (0 = lowest priority, 1 = highest), like PERCENTRANK in Excel.
-- Fewer stations per car means a bigger gap, so that indicator is sorted in descending order.
ranks AS (
    SELECT
        *,
        PERCENT_RANK() OVER (ORDER BY stations_per_10k_cars DESC) AS rank_gap,
        PERCENT_RANK() OVER (ORDER BY no_parking_share_2021) AS rank_no_parking,
        PERCENT_RANK() OVER (ORDER BY new_cars_per_1000_people) AS rank_new_cars
    FROM indicators
)

-- RANK gives provinces with the same score the same place; rounding removes floating point noise
SELECT
    RANK() OVER (ORDER BY ROUND

,priority_rank,province,region,priority_score
0,1,İzmir,Ege,0.80
1,2,Kırşehir,Orta Anadolu,0.75
2,3,Hatay,Akdeniz,0.72
3,4,Adıyaman,Güneydoğu Anadolu,0.71
4,5,Denizli,Ege,0.71
5,6,Manisa,Ege,0.70
6,6,İstanbul,İstanbul,0.70
7,8,Elazığ,Ortadoğu Anadolu,0.69
8,9,Erzincan,Kuzeydoğu Anadolu,0.66
9,10,Balıkesir,Batı Marmara,0.66


In [9]:
# Compare with the ranking saved by notebook 06
ranking = pd.read_csv(PROCESSED + "province_priority_ranking.csv").head(20)
comparison = top20_sql[["priority_rank", "province", "priority_score"]].merge(
    ranking[["province", "priority_rank", "priority_score"]], on="province", how="outer", suffixes=("_sql", "_pandas"))
print("Same 20 provinces:", comparison[["priority_rank_sql", "priority_rank_pandas"]].notna().all().all())
print("Same rank for every province:", (comparison["priority_rank_sql"] == comparison["priority_rank_pandas"]).all())
print(f"Largest score difference: {(comparison['priority_score_sql'] - comparison['priority_score_pandas']).abs().max():.3f}")
comparison.sort_values("priority_rank_sql")

Same 20 provinces: True
Same rank for every province: True
Largest score difference: 0.008


,priority_rank_sql,province,priority_score_sql,priority_rank_pandas,priority_score_pandas
19,1,İzmir,0.80,1,0.802
10,2,Kırşehir,0.75,2,0.757
8,3,Hatay,0.72,3,0.724
1,4,Adıyaman,0.71,4,0.716
5,5,Denizli,0.71,5,0.712
18,6,İstanbul,0.70,6,0.704
11,6,Manisa,0.70,6,0.704
6,8,Elazığ,0.69,8,0.691
7,9,Erzincan,0.66,9,0.667
3,10,Balıkesir,0.66,10,0.663


## Result

All four results are the same in SQL and in pandas: the monthly electric cars per socket, the regional ratios and Türkiye totals, the five development groups (same sizes and same values) and the top 20 priority provinces with the same rank for every province. The only difference is in the priority scores themselves (at most 0.008), which comes from the slightly different percentile rank formula explained above. The conclusions of the project do not depend on how the numbers were calculated.

This check found one small issue, now fixed in notebook 06: three provinces with exactly the same score (Sinop, Ordu, Zonguldak) were given different places because of floating point noise. They now share 12th place.